# 01) Prepare CodeBERT vulnerability-detection dataset (CVEfixes)

This notebook extracts a Python subset from the `CVEfixes_v1.0.0` SQL dump and builds a binary classification dataset:
- label `1` = vulnerable (method_change.before_change == True)
- label `0` = not vulnerable (method_change.before_change == False)

It outputs: `codebert_dataset.csv`

# 01) Prepare CodeBERT vulnerability-detection dataset (CVEfixes)

This notebook extracts a Python subset from the `CVEfixes_v1.0.0` SQL dump and builds a binary classification dataset:
- label `1` = vulnerable (method_change.before_change == True)
- label `0` = not vulnerable (method_change.before_change == False)

It outputs: `codebert_dataset.csv`

In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

DATASET_DIR = Path("/kaggle/working")  # Kaggle working dir
CVEFIXES_DIR = Path("CVEfixes_v1.0.0")  # <-- adjust if needed (e.g., /kaggle/input/<dataset>/CVEfixes_v1.0.0)
SQL_GZ_PATH = CVEFIXES_DIR / "Data" / "CVEfixes-2021-06-09.sql.gz"
SQLITE_PATH = DATASET_DIR / "CVEfixes.db"

MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "5000"))
PYTHON_ONLY = True

DATASET_DIR.mkdir(parents=True, exist_ok=True)
print("SQL_GZ_PATH:", SQL_GZ_PATH)
print("SQLITE_PATH:", SQLITE_PATH)

In [ ]:
# Build SQLite DB (takes time; do once per session / when db is missing)
if not SQLITE_PATH.exists():
    if not SQL_GZ_PATH.exists():
        raise FileNotFoundError(f"Missing SQL dump: {SQL_GZ_PATH}")
    # Use system sqlite3 + gunzip stream.
    !python -c "print('Creating SQLite DB...')"
    !bash -lc "gunzip -c '{SQL_GZ_PATH}' | sqlite3 '{SQLITE_PATH}'"

print("SQLite ready.")

In [ ]:
conn = sqlite3.connect(str(SQLITE_PATH))

# Binary labels from method_change.before_change boolean.
# Join path ensures we only keep Python code.
lang_filter = "='Python'" if PYTHON_ONLY else "is not null"

sql = f"""
SELECT
  m.code AS code,
  CAST(m.before_change AS INTEGER) AS label,
  cc.cwe_id AS cwe_id
FROM method_change m
JOIN file_change f ON m.file_change_id = f.file_change_id
LEFT JOIN commits c ON f.hash = c.hash
LEFT JOIN fixes fx ON fx.hash = c.hash
LEFT JOIN cwe_classification cc ON cc.cve_id = fx.cve_id
WHERE f.programming_language {lang_filter}
  AND m.code IS NOT NULL
LIMIT {MAX_SAMPLES}
"""

df = pd.read_sql_query(sql, conn)
conn.close()

# Clean up
df = df.dropna(subset=["code", "label"])
df["label"] = df["label"].astype(int)

print(df.head())
print("Label distribution:")
print(df["label"].value_counts())

out_path = DATASET_DIR / "codebert_dataset.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
out_path